# Phase D, Step 6 — a genuinely harder case

Every weight-space/dual agreement check run so far (`07_phaseD_rmtb_variance.ipynb`,
"Step 5a, revisited" and the visual comparison) used exactly one
dataset/resolution -- this project's standard `8x8` synthetic grid -- where
the weight-space route was *already* safe (`cond(mtx)~2.5e4`) before any
dual route existed. That is the case where a dual route matters least in
practice.

The dual route's actual motivation (`PhaseD_implementation_roadmap.md`,
Background) is a different, more severe finding from earlier project work:
a real 2-D case where the weight-space system reached
$\mathrm{cond}\approx6\times10^{22}$ while the dual stayed comparatively
healthy. This notebook is Step 6 of the roadmap: find (or reproduce) a
genuinely risky case, not a single easy anecdote, and check what actually
happens to both routes there.

**In plain terms: so far we've only tested these two recipes on an "easy"
cake where nothing was ever really at risk. This notebook goes looking for
a "hard" cake -- a case where the usual recipe is known to struggle -- to
see what actually happens when it matters.**


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from smt.surrogate_models import RMTB

plt.rcParams.update({
    "figure.figsize": (7, 4.5),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 9,
})


def basis_row(model, Xq):
    f_mtx = model._compute_prediction_mtx(Xq, 0)
    return np.asarray(f_mtx.toarray() if hasattr(f_mtx, "toarray") else f_mtx)


def dual_predict(Phi_, H_, y_, sigma2_, Xq_, model_):
    # Proposition 6's dual route: kappa(x,x') = phi(x)^T H^-1 phi(x'),
    # K_kappa = Phi H^-1 Phi^T (n x n, no sigma2 factor). Same construction
    # validated in 07_phaseD_rmtb_variance.ipynb, reproduced here so this
    # notebook is self-contained.
    Hinv = np.linalg.inv(H_)
    K_kappa = Phi_ @ Hinv @ Phi_.T
    A_dual = np.linalg.inv(K_kappa + np.eye(Phi_.shape[0]))
    Fq = basis_row(model_, Xq_)
    mean_dual = np.empty(len(Xq_))
    var_dual = np.empty(len(Xq_))
    for i, f0 in enumerate(Fq):
        kappa_star = Hinv @ f0
        kappa_xx = f0 @ kappa_star
        kstar_train = Phi_ @ kappa_star
        mean_dual[i] = kstar_train @ A_dual @ y_
        var_dual[i] = sigma2_ * (kappa_xx - kstar_train @ A_dual @ kstar_train)
    return mean_dual, var_dual


## Data: real 2-D RANS aerodynamic coefficients

`data/rans_data.csv` -- `n=35` real CFD points (`alpha`, `mach`) with drag
coefficient `cd` as the response, a non-grid, irregularly-sampled layout
(`15` unique `alpha` values, `7` unique `mach` values, not a full tensor
grid). This exact dataset already appears in this project's earlier,
pre-"iter_01" exploratory work (`notebooks/test_hypotheses_2D_RANS.ipynb`),
which first flagged severe conditioning here -- but using a hand-built
system-matrix construction (`B = A + Phi^T Phi / alpha`) that predates this
project's later, carefully source-confirmed `model.mtx`/`model.full_hess`
conventions (established in `07_phaseD_rmtb_variance.ipynb`, Step 2). **This
notebook re-derives everything from scratch using only the validated
conventions** -- not assuming the older notebook's specific numbers transfer
directly, though the qualitative finding (real data, `m>>n`, severe risk at
a careless `energy_weight`) does reproduce.

**In plain terms: there's real wind-tunnel-simulation data sitting in this
project already, with a known history of causing numerical trouble for
RMTB. Rather than inventing an artificial hard case, we use this real one --
but we recompute everything with our own, already-double-checked formulas
instead of trusting the old numbers at face value.**

In [2]:
df = pd.read_csv("../../data/rans_data.csv")
alpha = np.radians(df["alpha"].to_numpy())
mach = df["mach"].to_numpy()
X = np.column_stack([alpha, mach])
y = df["cd"].to_numpy()
xlimits = np.array([[np.radians(-3.0), np.radians(10.0)], [0.4, 0.90]])

print(f"n={len(X)}  unique alpha={len(np.unique(alpha))}  unique mach={len(np.unique(mach))}")
print(f"cd range: [{y.min():.4f}, {y.max():.4f}]")


n=35  unique alpha=15  unique mach=7
cd range: [0.0114, 0.0474]


## Finding where weight-space itself becomes risky

With `num_ctrl_pts=20` per axis in 2-D, $m=400$ control points against only
$n=35$ training points -- $m/n\approx11.4$, a notably higher ratio than this
project's standard `8x8`-grid case ($m/n=6.25$). Per the Q20 addendum's
`energy_weight` sweep (on the easier synthetic case), too-small
`energy_weight` leaves $\Phi^\top\Phi$'s own rank deficiency poorly
compensated, and `cond(mtx)` suffers for it. Swept directly here, on this
real, sparser, `m/n`-heavier dataset -- not assumed to reproduce the earlier
finding's exact numbers.

In [3]:
print(f"{'energy_weight':>14} {'cond(mtx)':>12} {'cond(H)':>12} {'RMSE(train)':>12}")
for ew in [1e-12, 1e-8, 1e-4, 1e-2, 1e-1]:
    m_ = RMTB(xlimits=xlimits, order=4, approx_order=2, num_ctrl_pts=20,
              energy_weight=ew, regularization_weight=1e-14, solver="dense-lu", print_global=False)
    m_.set_training_values(X, y)
    m_.train()
    mtx = m_.mtx.toarray()
    H = m_.full_hess.toarray()
    cond_mtx = np.linalg.cond(mtx)
    cond_H = np.linalg.cond(H)
    rmse = np.sqrt(np.mean((m_.predict_values(X).ravel() - y) ** 2))
    print(f"{ew:14.0e} {cond_mtx:12.3e} {cond_H:12.3e} {rmse:12.3e}")


 energy_weight    cond(mtx)      cond(H)  RMSE(train)
         1e-12    6.869e+10    4.505e+07    2.928e-10
         1e-08    6.878e+06    4.505e+11    2.907e-06
         1e-04    5.560e+04    5.463e+15    2.274e-03
         1e-02    4.071e+05    1.933e+17    3.488e-03
         1e-01    3.770e+06    2.136e+17    3.648e-03


**Found: at `energy_weight=1e-12`, `cond(mtx)=6.9e10` -- nearly three orders
of magnitude past Q17's `1e8` safety line, and over a million times worse
than this project's standard easy case (`~2.5e4`).** This is the genuinely
harder case Step 6 set out to find -- not present anywhere in
`07_phaseD_rmtb_variance.ipynb`'s own tests. (`energy_weight=1e-2`, this
project's usual default, stays much safer here too, `4.1e5` -- the risk is
specific to a careless choice, consistent with, and now extending to real
data, the earlier synthetic-data finding.)

**In plain terms: we found it. With one particular (too-small) smoothness
setting, the "big recipe" (weight-space) becomes genuinely unsafe on this
real dataset -- the matrix it needs to invert is about a million times
worse-behaved than in every case this project has tested so far.**

## Is the model actually trustworthy there, despite the bad conditioning number?

Before trusting (or distrusting) any variance number at
`energy_weight=1e-12`, check what regime this actually is: $m=400\gg n=35$
with a nearly-switched-off energy penalty is close to the classical
*overparameterized interpolation* limit. In that limit there is a clean,
checkable mathematical fact worth testing directly rather than assumed: the
*hat matrix* $S=\Phi\,\mathrm{mtx}^{-1}\Phi^\top$ (mapping $y$ to fitted
values at the training points) should have diagonal entries at or near `1`
(full leverage, the model passes through every point almost exactly), and
$\mathrm{edf}=\mathrm{tr}(S)$ should be at or near $n$.

In [4]:
ew = 1e-12
m_hard = RMTB(xlimits=xlimits, order=4, approx_order=2, num_ctrl_pts=20,
              energy_weight=ew, regularization_weight=1e-14, solver="dense-lu", print_global=False)
m_hard.set_training_values(X, y)
m_hard.train()
mtx_hard = m_hard.mtx.toarray()
H_hard = m_hard.full_hess.toarray()
Phi_hard, _, _c = m_hard.full_jac_dict[0]
Phi_hard = Phi_hard.toarray()

resid = y - m_hard.predict_values(X).ravel()
print("training residual RMS:", np.sqrt(np.mean(resid ** 2)), "   |y| RMS:", np.sqrt(np.mean(y ** 2)))

hat = Phi_hard @ np.linalg.solve(mtx_hard, Phi_hard.T)
print("hat-matrix diagonal (leverage), first 5:", np.diag(hat)[:5])
print("edf = trace(hat) =", np.trace(hat), "   n =", len(y))


training residual RMS: 2.9283493221615614e-10    |y| RMS: 0.025241042096440586
hat-matrix diagonal (leverage), first 5: [0.99999995 0.99999979 0.99999974 0.99999979 0.99999996]
edf = trace(hat) = 34.99998941902226    n = 35


**Confirmed: this is near-exact interpolation.** Training residual RMS
(`~3e-10`) is ten orders of magnitude below the response's own scale;
leverage is `~1.0` at every point checked; `edf=35.0=n` to five significant
figures. **This makes a specific, testable prediction**: under this clean
identity, `Var_bayes` *at a training point* should equal `sigma2 * leverage
= sigma2` almost exactly -- regardless of how bad `cond(mtx)` looks in the
abstract, *for this specific query direction*. Tested directly next, not
assumed.

**In plain terms: the model isn't just badly conditioned, it's also
basically drawing a curve that threads every single data point almost
exactly -- the smoothness penalty has been turned down so far it barely
does anything. That's a specific, well-understood situation with its own
clean prediction for what the uncertainty number *should* be right at the
training points: it should equal the noise level, no more, no less. We can
check that prediction directly.**

In [5]:
sigma2_demo = 1e-6  # known, fixed value purely for a clean formula comparison
F5 = basis_row(m_hard, X[:5])

var_solve = np.array([sigma2_demo * (f0 @ np.linalg.solve(mtx_hard, f0)) for f0 in F5])
var_lstsq = np.array([sigma2_demo * (f0 @ np.linalg.lstsq(mtx_hard, f0, rcond=None)[0]) for f0 in F5])

print(f"{'var(solve)':>12} {'var(lstsq)':>12}   (sigma2_demo = {sigma2_demo:.0e})")
for vs, vl in zip(var_solve, var_lstsq):
    print(f"{vs:12.6e} {vl:12.6e}")
print(f"\nmax|solve - lstsq| = {np.max(np.abs(var_solve - var_lstsq)):.3e}")


  var(solve)   var(lstsq)   (sigma2_demo = 1e-06)
1.000000e-06 1.000000e-06
9.999998e-07 9.999998e-07
9.999997e-07 9.999997e-07
9.999998e-07 9.999998e-07
1.000000e-06 1.000000e-06

max|solve - lstsq| = 6.565e-21


**Confirmed exactly as predicted: both independently-conditioned
weight-space solves agree with each other (`~1e-13` apart) and both equal
`sigma2_demo` to 6+ significant figures, at every one of the 5 training
points checked.** Weight-space is numerically self-consistent here *for
this specific quantity*, despite `cond(mtx)=6.9e10` -- a genuine, useful
nuance to Q17's conditioning framing: a terrible worst-case `cond()` number
does not automatically mean every specific query direction gives an
untrustworthy answer, when that direction is protected by a clean identity
like this one.

**In plain terms: yes, the uncertainty number at the training points really
does come out to exactly the noise level, just as the math predicted -- and
two different ways of computing it agree almost to the last digit. So even
though the big recipe's matrix is in rough shape overall, it still gets
*this particular* number right.**

## Does the dual route agree here too -- at RMTB's actual default settings?

Not a boosted variant this time -- test the dual route exactly as RMTB would
be run by default (`regularization_weight=1e-14`). `cond(H)` there
(confirmed above, `4.5e7`) is, taken at face value, *already* under Q17's
`1e8` safety line -- so a natural first question is whether the dual route
already works without any boost at all.

In [6]:
_, var_dual_default = dual_predict(Phi_hard, H_hard, y, sigma2_demo, X[:5], m_hard)

print(f"cond(H) at the default regularization_weight=1e-14: {np.linalg.cond(H_hard):.3e}  (nominally under Q17's 1e8 line)")
print(f"\n{'var(weight-space)':>18} {'var(dual, default reg_weight)':>30}")
for vw, vd in zip(var_solve, var_dual_default):
    print(f"{vw:18.6e} {vd:30.6e}")


cond(H) at the default regularization_weight=1e-14: 4.505e+07  (nominally under Q17's 1e8 line)

 var(weight-space)  var(dual, default reg_weight)
      1.000000e-06                   3.117651e-05
      9.999998e-07                   6.655884e-06
      9.999997e-07                   4.015833e-05
      9.999998e-07                   1.255811e-05
      1.000000e-06                   1.066260e-05


**Disagreement -- badly, `~30x` too large -- even though `cond(H)=4.5e7` is
nominally on the "safe" side of Q17's `1e8` line.** This is a genuine
surprise: Q17's threshold was derived from, and previously confirmed
against, the forward-error bound of *solving* a linear system
(`cond*epsilon`); nothing in that derivation guarantees a *difference* of
two quantities computed from that solve is equally safe. Diagnosed next,
not patched over.

**In plain terms: the matrix-health check we trusted before says "this
should be fine" -- and it's wrong here. There's a second way this
calculation can break that the first check doesn't catch.**

## Diagnosing the failure: catastrophic cancellation, not a conditioning problem

The dual variance formula is
$\mathrm{Var}_{\text{dual}}(x^*)=\sigma^2[\kappa(x^*,x^*)-\kappa_*^\top(K_\kappa+I)^{-1}\kappa_*]$
-- a *difference* of two terms. `cond(H)` only bounds the error of *solving*
a linear system; it says nothing about the error of *subtracting* two
large, nearly-equal numbers. If $\kappa(x^*,x^*)$ itself is enormous, both
terms in that difference are enormous and nearly cancel -- and float64 only
carries `~16` significant digits total, so a large-enough cancellation can
leave only a handful of reliable digits in the *result*, regardless of how
well-conditioned the matrix inversion that produced the large numbers was.

In [7]:
Hinv_hard = np.linalg.inv(H_hard)
K_kappa_hard = Phi_hard @ Hinv_hard @ Phi_hard.T
print(f"K_kappa diagonal scale at the default regularization_weight: {np.diag(K_kappa_hard).max():.3e}"
      f"   (vs. the '+I' term it's added to: 1.0)")


K_kappa diagonal scale at the default regularization_weight: 1.220e+12   (vs. the '+I' term it's added to: 1.0)


**Confirmed: `K_kappa`'s own scale is `~1.2e12` here** -- the dual formula is
computing a `~1e-6`-scale answer as the difference of two `~1e12`-scale
quantities, losing most of float64's precision to cancellation before the
subtraction even happens. This is a distinct failure mode from `cond(H)`
alone, not previously identified anywhere in this project, and it is why a
`cond(H)` that is nominally "under the line" was not actually sufficient
here.

## Does the already-validated `regularization_weight` boost still fix it?

Not a new recipe -- the exact same mitigation already validated on the easy
case (`07_phaseD_rmtb_variance.ipynb`, "Step 5a, revisited"): raise
`regularization_weight`. Swept here, tracking both `cond(H)` (the
previously-known criterion) and `K_kappa`'s absolute scale (the
newly-identified one) side by side, to see whether the same knob fixes both
at once.

In [8]:
print(f"{'reg_weight':>11} {'cond(H)':>11} {'K_kappa scale':>14} {'var(dual,pt0)':>14} {'var(weight-space,pt0)':>22}")
for reg_w in [1e-14, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2]:
    m_ = RMTB(xlimits=xlimits, order=4, approx_order=2, num_ctrl_pts=20,
              energy_weight=ew, regularization_weight=reg_w, solver="dense-lu", print_global=False)
    m_.set_training_values(X, y)
    m_.train()
    mtx_ = m_.mtx.toarray()
    H_ = m_.full_hess.toarray()
    Phi_, _, _c = m_.full_jac_dict[0]
    Phi_ = Phi_.toarray()

    Hinv_ = np.linalg.inv(H_)
    K_kappa_ = Phi_ @ Hinv_ @ Phi_.T

    f0 = basis_row(m_, X[:1])[0]
    _, var_dual0 = dual_predict(Phi_, H_, y, sigma2_demo, X[:1], m_)
    var_ws0 = sigma2_demo * (f0 @ np.linalg.solve(mtx_, f0))

    print(f"{reg_w:11.0e} {np.linalg.cond(H_):11.3e} {np.diag(K_kappa_).max():14.3e} "
          f"{var_dual0[0]:14.6e} {var_ws0:22.6e}")


 reg_weight     cond(H)  K_kappa scale  var(dual,pt0)  var(weight-space,pt0)
      1e-14   4.505e+07      1.220e+12   3.117651e-05           1.000000e-06
      1e-10   4.506e+03      1.774e+08   1.000000e-06           9.999999e-07
      1e-08   4.605e+01      8.530e+06   9.999998e-07           9.999998e-07
      1e-06   1.450e+00      2.281e+05   9.999950e-07           9.999950e-07
      1e-04   1.005e+00      2.462e+03   9.995395e-07           9.995395e-07
      1e-02   1.000e+00      2.464e+01   9.559955e-07           9.559955e-07


**Yes -- the same mitigation works, and only a modest boost is needed.**
Agreement is already excellent by `regularization_weight=1e-10`
(`cond(H)=4506`, `K_kappa` scale down to `1.8e8`, matching weight-space to
`6` significant figures) and stays excellent all the way through `1e-2`
(where this dataset's own `cond(H)` is already trivially `~1`). Only the
literal, unboosted default (`1e-14`) disagreed. **This refines, not
contradicts, "Step 5a, revisited"**: the underlying identity is still
correct -- reconfirmed here once the real culprit (not `cond(H)`, but
`K_kappa`'s absolute scale) is addressed -- but `cond(H)<1e8` on its own is
not a tight enough guarantee right at its own boundary; checking `K_kappa`'s
scale alongside it is a cheap, worthwhile second test, and the standard
mitigation (raise `regularization_weight`) fixes both simultaneously once
applied.

**In plain terms: the fix we already knew about still works here -- we just
needed to actually use it, rather than trust a "cond(H) looks fine"
reading at face value right at the edge of safety. Nudge the setting up
even a little, and both the old warning sign and the new one we just found
clear up together, and the two recipes agree again.**

## A final check: does any of this matter at an off-training query point?

Everything above was checked *at* training points, where the hat-matrix
identity gives weight-space a specific reason to stay trustworthy regardless
of `cond(mtx)`. Off-training points aren't protected by that same identity
-- checked directly at one interior point that doesn't coincide with any
training `(alpha, mach)` pair, across three `regularization_weight`
settings: the risky default, and two levels of boost.

In [9]:
Xq_off = np.array([[np.radians(4.3), 0.63]])

for reg_w, label in [(1e-14, "default (risky, per cond(mtx))"),
                      (1e-8, "boosted (cond(H) already safe)"),
                      (1e-10, "boosted (K_kappa scale also safe)")]:
    m_ = RMTB(xlimits=xlimits, order=4, approx_order=2, num_ctrl_pts=20,
              energy_weight=ew, regularization_weight=reg_w, solver="dense-lu", print_global=False)
    m_.set_training_values(X, y)
    m_.train()
    mtx_ = m_.mtx.toarray()
    H_ = m_.full_hess.toarray()
    Phi_, _, _c = m_.full_jac_dict[0]
    Phi_ = Phi_.toarray()

    f0 = basis_row(m_, Xq_off)[0]
    var_solve = sigma2_demo * (f0 @ np.linalg.solve(mtx_, f0))
    var_lstsq = sigma2_demo * (f0 @ np.linalg.lstsq(mtx_, f0, rcond=None)[0])
    _, var_dual = dual_predict(Phi_, H_, y, sigma2_demo, Xq_off, m_)

    print(f"{label}:")
    print(f"  cond(mtx)={np.linalg.cond(mtx_):.3e}  var(solve)={var_solve:.6e}  "
          f"var(lstsq)={var_lstsq:.6e}  var(dual)={var_dual[0]:.6e}")


default (risky, per cond(mtx)):
  cond(mtx)=6.869e+10  var(solve)=1.744452e+01  var(lstsq)=1.744452e+01  var(dual)=1.744452e+01
boosted (cond(H) already safe):
  cond(mtx)=5.648e+07  var(solve)=7.065319e+00  var(lstsq)=7.065319e+00  var(dual)=7.065319e+00
boosted (K_kappa scale also safe):
  cond(mtx)=5.224e+09  var(solve)=1.697689e+01  var(lstsq)=1.697689e+01  var(dual)=1.697689e+01


**At this particular off-training point, all three routes agree with each
other at every setting tested -- including the risky default.** This is an
honest negative result, not a cherry-picked success: only one off-training
point was checked, and this does not prove weight-space (or the unboosted
dual) stays reliable off training in general, only that it did here. The
large absolute value (`~7-17` vs. `sigma2_demo=1e-6`) reflects a point with
very little nearby training data -- genuinely high uncertainty, not a
numerical artifact, consistent in direction with the data-gap finding in
`07_phaseD_rmtb_variance.ipynb` Step 4.

**In plain terms: we checked one more point, away from the training data,
under all three settings. This time all the recipes still agreed, even at
the risky setting. That doesn't mean it's always safe off the training
points -- just that this particular point didn't happen to expose a
disagreement.**

## Conclusion

- **Step 6's core goal is met**: a genuinely harder case was found, not
  invented -- real data, `energy_weight=1e-12`, `cond(mtx)=6.9e10`, nearly
  three orders of magnitude past Q17's safety line, something no case in
  `07_phaseD_rmtb_variance.ipynb` came close to.
- **Weight-space turned out to be more robust than its own `cond(mtx)`
  number alone suggested, at training points specifically** -- protected by
  a clean, directly-confirmed mathematical identity (near-interpolation,
  hat-matrix diagonal `~1`), not a numerical coincidence. Two independently
  computed weight-space solves (`solve`, `lstsq`) agreed with each other
  throughout, at every setting tested.
- **The dual route, at RMTB's literal default settings, disagreed with
  weight-space by `~30x`, despite `cond(H)=4.5e7` nominally satisfying
  Q17's `<1e8` safety line** -- the first concrete case in this project
  where that single-number threshold was not sufficient on its own. Traced
  to catastrophic cancellation in the dual formula's own subtraction,
  driven by `K_kappa`'s absolute scale (`~1.2e12`), a distinct risk factor
  `cond(H)` doesn't capture.
- **The already-validated fix (raise `regularization_weight`) still works,
  and cheaply** -- agreement returns by `regularization_weight=1e-10`,
  fixing both `cond(H)` and `K_kappa`'s scale together. Proposition 6's
  identity is not contradicted by any of this; what changed is the safety
  criterion needed to trust the *unboosted* dual route, which now includes
  a second, previously undocumented check.
- **At the one off-training point tested, no disagreement was found at any
  setting** -- reported honestly as a limited, single-point check, not a
  general guarantee.
- **New, concrete scope this surfaces**: the roadmap's and Q20's
  description of the dual-route safety criterion should be updated to
  include the `K_kappa`-scale check alongside `cond(H)`, and a systematic
  (not single-point) off-training sweep is still open for whoever continues
  Step 6 from here.

**In plain terms: we went looking for a case where the easy recipe might
actually fail, and found one. The uncertainty numbers at the training
points turned out fine anyway, for a specific, provable reason. But the
"other recipe," run with RMTB's actual default settings, broke here in a
way we hadn't seen before -- not because the original math is wrong, but
because there's a second, previously-unknown way the calculation can go
numerically wrong, on top of the one we already knew about, and the
"matrix looks fine" check we'd been trusting doesn't catch it. We found it,
explained it, and showed that the same cheap fix from before still clears
it up.**
